In [ ]:
!pip install requests


In [ ]:
import requests

def fetch_weather():
    url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/Jaipur?unitGroup=metric&key=9FLZQ4N4M5KWSYLMLJ4SUG7PX&contentType=json"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        current = data.get('currentConditions', {})

        print("📍 Weather Data for Jaipur")
        print(f"🗓️ Date: {data.get('days', [{}])[0].get('datetime', 'N/A')}")
        print(f"🌡️ Temperature: {current.get('temp', 'N/A')} °C")
        print(f"💧 Humidity: {current.get('humidity', 'N/A')}%")
        print(f"🌤️ Conditions: {current.get('conditions', 'N/A')}")
        print(f"🌬️ Wind Speed: {current.get('windspeed', 'N/A')} km/h")

    except requests.exceptions.RequestException as e:
        print("❌ Error fetching weather data:", e)

# Run it
fetch_weather()


📍 Weather Data for Jaipur
🗓️ Date: 2025-06-18
🌡️ Temperature: 27.0 °C
💧 Humidity: 83.7%
🌤️ Conditions: Rain, Partially cloudy
🌬️ Wind Speed: 9.4 km/h


# Raw Jason Format

In [ ]:
import requests
import json

def fetch_weather_json_selected():
    url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/Jaipur?unitGroup=metric&key=9FLZQ4N4M5KWSYLMLJ4SUG7PX&contentType=json"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        current = data.get('currentConditions', {})

        selected_data = {
            "location": data.get("resolvedAddress", "Jaipur"),
            "date": data.get("days", [{}])[0].get("datetime", "N/A"),
            "temperature_celsius": current.get("temp", "N/A"),
            "humidity_percent": current.get("humidity", "N/A"),
            "conditions": current.get("conditions", "N/A"),
            "wind_speed_kmph": current.get("windspeed", "N/A")
        }

        print(json.dumps(selected_data, indent=4))

    except requests.exceptions.RequestException as e:
        print("❌ Error fetching weather data:", e)

# Run the selected version
fetch_weather_json_selected()


{
    "location": "Jaipur, RJ, India",
    "date": "2025-06-18",
    "temperature_celsius": 28.0,
    "humidity_percent": 78.9,
    "conditions": "Rain, Overcast",
    "wind_speed_kmph": 7.6
}


# CSV Format

In [ ]:
import requests
import csv

def fetch_weather_to_csv():
    url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/Jaipur?unitGroup=metric&key=9FLZQ4N4M5KWSYLMLJ4SUG7PX&contentType=json"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        current = data.get('currentConditions', {})
        selected_data = {
            "location": data.get("resolvedAddress", "Jaipur"),
            "date": data.get("days", [{}])[0].get("datetime", "N/A"),
            "temperature_celsius": current.get("temp", "N/A"),
            "humidity_percent": current.get("humidity", "N/A"),
            "conditions": current.get("conditions", "N/A"),
            "wind_speed_kmph": current.get("windspeed", "N/A")
        }

        # Write to CSV
        with open("jaipur_weather.csv", mode="w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=selected_data.keys())
            writer.writeheader()
            writer.writerow(selected_data)

        print("✅ Weather data saved to 'jaipur_weather.csv'")

    except requests.exceptions.RequestException as e:
        print("❌ Error fetching weather data:", e)

# Run the function
fetch_weather_to_csv()


✅ Weather data saved to 'jaipur_weather.csv'


# Gobbi

In [ ]:
pip install requests pandas


In [ ]:
import requests
import json

api_key = ""
target_url = "https://www.zomato.com/jaipur/restaurants"

payload = {
    "prompt": f"Visit {target_url} and return the page title and first <h1> as JSON",
    "output_schema": {
        "type": "object",
        "properties": {
            "page_title": {"type": "string"},
            "first_h1": {"type": "string"}
        },
        "required": ["page_title", "first_h1"]
    },
    "wait": 300
}

response = requests.post(
    "https://gobii.ai/api/v1/tasks/browser-use/",
    headers={
        "X-Api-Key": api_key,
        "Content-Type": "application/json"
    },
    data=json.dumps(payload)
)

# ✅ Fix: check for 200 OR 201
if response.status_code in [200, 201]:
    result = response.json().get("result")
    print(json.dumps(result, indent=4))
else:
    print("❌ Error:", response.status_code, response.text)

{
    "first_h1": "Access Denied",
    "page_title": "Access Denied"
}


# 15 Days data

In [ ]:
import requests
import csv
from datetime import datetime, timedelta

def fetch_last_15_days_weather_csv():
    # Date range
    end_date = datetime.today()
    start_date = end_date - timedelta(days=29)

    start = start_date.strftime('%Y-%m-%d')
    end = end_date.strftime('%Y-%m-%d')

    url = f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/Jaipur/{start}/{end}?unitGroup=metric&key=9FLZQ4N4M5KWSYLMLJ4SUG7PX&contentType=json"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Extract relevant data
        rows = []
        for day in data.get("days", []):
            rows.append({
                "Date": day.get("datetime"),
                "Temperature (°C)": day.get("temp", "N/A"),
                "Humidity (%)": day.get("humidity", "N/A"),
                "Conditions": day.get("conditions", "N/A"),
                "Wind Speed (km/h)": day.get("windspeed", "N/A")
            })

        # CSV file writing
        filename = "jaipur_last_15_days_weather.csv"
        with open(filename, mode="w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)

        print(f"✅ CSV saved as '{filename}'")

    except requests.exceptions.RequestException as e:
        print("❌ Error fetching weather data:", e)

# Run it
fetch_last_15_days_weather_csv()


✅ CSV saved as 'jaipur_last_15_days_weather.csv'
